In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# <span style="color:red">ch9.01_vector_Embedding_Model성능비교</span>

# 문장 -> 벡터(1차원 숫자 배열 [8.1, 9.1, 2, 5, 4, 3, ...])

- openAi API : https://platform.openai.com/ 의 키를 .env(OPENAI_API_KEY)에 등록
- upstage : https://console.upstage.ai/ 의 키를 .env(UPSTAGE_API_KEY)에 등록

# 1. 환경변수 load

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# 2. 유사도 계산하는 방법 : https://www.pinecone.io/learn/vector-similarity
    - 1. 유클리드 거리 : 두 벡터간의 거리가 가까운지
    - 2. 코사인유사도 : 두 벡터간 방향이 유사한지
    - 3. dot product : 두 벡터간의 곱을 사용하여 거리와 방향을 모두 고려

In [13]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2) # 행 벡터의 곱
    norm_vec1 = np.linalg.norm(vec1) # 벡터의 길이
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)

# 3. openAi API의 embedding model 사용

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
# text-embedding-3-large
response = openai_client.embeddings.create(
    input="king",
    model="text-embedding-3-large"
)

In [10]:
import numpy as np
king_vector = np.array(response.data[0].embedding)
print(king_vector.shape)
print(king_vector)

(3072,)
[ 0.01040417  0.02499519 -0.0014776  ...  0.00835009  0.01049861
 -0.00254005]


In [19]:
queen_response = openai_client.embeddings.create(
    input="queen",
    model="text-embedding-3-large"
)

In [12]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector.shape)
print(queen_vector)

(3072,)
[-0.01385735  0.0008602  -0.0167823  ...  0.00017693  0.01159847
  0.00638929]


In [14]:
king_queen_similarity = cosine_similarity(king_vector, queen_vector)
print(king_queen_similarity)

0.5552268369726675


In [18]:
slave_response = openai_client.embeddings.create(
    input="slave",
    model="text-embedding-3-large"
)

In [20]:
slave_vector = np.array(slave_response.data[0].embedding)
print(slave_vector.shape)
print(slave_vector)

(3072,)
[-0.01999537  0.00620363  0.01191717 ...  0.00094749 -0.02679118
 -0.0058524 ]


In [21]:
king_slave_similarity = cosine_similarity(king_vector, slave_vector)
print("king과 slave 유사도 :", king_slave_similarity)

king과 slave 유사도 : 0.2947745074537996


### 한국어 문장을 벡터로 바꿔도 유사도는 비슷해야 할 듯

In [22]:
kor_king_response = openai_client.embeddings.create(
    input="왕",
    model="text-embedding-3-large"
)

In [23]:
kor_king_vector = np.array(kor_king_response.data[0].embedding)
print(kor_king_vector.shape)
print(kor_king_vector)

(3072,)
[-0.00595223  0.01159333 -0.01316932 ... -0.00357134  0.01323696
 -0.00083999]


In [24]:
kor_queen_response = openai_client.embeddings.create(
    input="여왕",
    model="text-embedding-3-large"
)

In [25]:
kor_queen_vector = np.array(kor_queen_response.data[0].embedding)
print(kor_queen_vector.shape)
print(kor_queen_vector)

(3072,)
[-0.01307151 -0.00921458 -0.00532257 ... -0.00482468 -0.00204418
  0.02035061]


In [26]:
kor_king_queen_similarity = cosine_similarity(kor_king_vector, kor_queen_vector)
print("한국어 왕과 여왕 유사도 :", kor_king_queen_similarity)

한국어 왕과 여왕 유사도 : 0.48733449549538954


In [27]:
kor_slave_response = openai_client.embeddings.create(
    input="거지",
    model="text-embedding-3-large"
)

In [28]:
kor_slave_vector = np.array(kor_slave_response.data[0].embedding)
print(kor_slave_vector.shape)
print(kor_slave_vector)

(3072,)
[-0.02400834 -0.02815736 -0.00371585 ...  0.01028707 -0.00947125
  0.03754314]


In [30]:
kor_king_slave_similarity = cosine_similarity(kor_king_vector, kor_slave_vector)
print("한국어 왕과 거지 유사도 :", kor_king_slave_similarity)

한국어 왕과 거지 유사도 : 0.2552452064791607


In [32]:
# king과 왕의 유사도
king_왕_similarity = cosine_similarity(king_vector, kor_king_vector)
print("king과 왕의 유사도 :", king_왕_similarity)

king과 왕의 유사도 : 0.5474873912140233


# 4. upstage의 embedding model 사용

- 한국어 embedding에는 openai보다 성능이 훨씬 좋다

In [33]:
from openai import OpenAI
import os
upstage_api_key = os.getenv('UPSTAGE_API_KEY')
upstage_client = OpenAI(
    api_key=upstage_api_key,
    base_url="https://api.upstage.ai/v1"
)

In [35]:
up_king_response = upstage_client.embeddings.create(
    input="king",
    model="embedding-query"
)

In [36]:
up_king_vector = np.array(up_king_response.data[0].embedding)
print(up_king_vector.shape)
print(up_king_vector)

(4096,)
[-0.01187134 -0.02058411 -0.00674438 ... -0.01082611  0.00244713
  0.01517487]


In [37]:
up_queen_response = upstage_client.embeddings.create(
    input="queen",
    model="embedding-query"
)

In [38]:
up_queen_vector = np.array(up_queen_response.data[0].embedding)
print(up_queen_vector.shape)
print(up_queen_vector)

(4096,)
[-0.0016222  -0.00952148 -0.00471878 ...  0.00985718 -0.00732803
  0.0259552 ]


In [39]:
up_king_queen_similarity = cosine_similarity(up_king_vector, up_queen_vector)
print("up king queen 유사도 :", up_king_queen_similarity)

up king queen 유사도 : 0.6277983746920601


In [40]:
up_kor_king_response = upstage_client.embeddings.create(
    input="왕",
    model="embedding-query"
)

In [41]:
up_kor_king_vector = np.array(up_kor_king_response.data[0].embedding)
print(up_kor_king_vector.shape)
print(up_kor_king_vector)

(4096,)
[-0.01210022 -0.02249146 -0.01314545 ... -0.00024557  0.00358391
  0.01416779]


In [42]:
up_king_왕_similarity = cosine_similarity(up_king_vector, up_kor_king_vector)
print("up king 왕 유사도 :", up_king_왕_similarity)

up king 왕 유사도 : 0.8521901935963604
